# MNIST MLP (HEX export) 中間結果デバッグ（1枚）
- 入力: u8 hex (784 lines)
- 重み: row-major int8 hex
- bias: int32 hex
- 量子化: meta.json の in_zp/out_zp/rq_s/rq_shift/is_relu_fused/out_scale

# 目的:
1) 乗算 (xc*w)
2) 累算 Σ
3) bias加算
4) requant (half-up)
5) +out_zp
6) Clip (relu_fusedなら下限=out_zp)
7) u8 出力


In [2]:
# ============================================================
# mlp_dump_lib.py (4-bit Variant - Correct Logic)
# - Calculation always runs for FULL cycles.
# - k_range only limits the PRINT output.
# ============================================================
from __future__ import annotations
from pathlib import Path
import json
import numpy as np

PE_NUM = 32

# --- 4-bit / 16-bit Constraints ---
INT16_MIN = -32768
INT16_MAX =  32767
UINT4_MIN =  0
UINT4_MAX =  15
INT4_MIN  = -8
INT4_MAX  =  7

# ---------- file loaders ----------
def load_meta(meta_path: Path) -> dict:
    with meta_path.open("r", encoding="utf-8") as f:
        return json.load(f)

def load_u4_hex_n(path: Path, n: int) -> np.ndarray:
    vals = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                vals.append(int(s, 16) & 0xF)
    if len(vals) != n:
        raise ValueError(f"input hex count mismatch: got {len(vals)} expected {n} ({path})")
    return np.array(vals, dtype=np.uint8)

def load_weight_rowmajor_int4(path: Path, out_ch: int, in_ch: int) -> np.ndarray:
    vals = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                v = int(s, 16) & 0xF
                if v >= 8: v -= 16
                vals.append(v)
    exp = out_ch * in_ch
    if len(vals) != exp:
        raise ValueError(f"weight hex count mismatch: got {len(vals)} expected {exp} ({path})")
    u = np.array(vals, dtype=np.int8)
    return u.reshape(out_ch, in_ch)

def load_bias_int16(path: Path, out_ch: int) -> np.ndarray:
    vals = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                v = int(s, 16) & 0xFFFF
                if v >= 32768: v -= 65536
                vals.append(v)
    if len(vals) != out_ch:
        raise ValueError(f"bias hex count mismatch: got {len(vals)} expected {out_ch} ({path})")
    return np.array(vals, dtype=np.int16)

# ---------- math helpers ----------
def clamp_i16_vec(x: np.ndarray) -> np.ndarray:
    return np.clip(x, INT16_MIN, INT16_MAX).astype(np.int16)

def sat_add_i16_vec(acc_i16: np.ndarray, val_i16: np.ndarray) -> np.ndarray:
    res_i32 = acc_i16.astype(np.int32) + val_i16.astype(np.int32)
    return clamp_i16_vec(res_i32)

def requant_hw_4bit(acc_i16: np.ndarray, rq_s: int, rq_shift: int) -> np.ndarray:
    mul = acc_i16.astype(np.int64) * int(rq_s)
    if rq_shift > 0:
        rnd_offset = (1 << (rq_shift - 1))
        mul += rnd_offset
        out = mul >> rq_shift
    else:
        out = mul
    return out.astype(np.int32)

def clip_to_u4_vec(val_i32: np.ndarray, out_zp: int) -> np.ndarray:
    qy = val_i32 + out_zp
    return np.clip(qy, UINT4_MIN, UINT4_MAX).astype(np.uint8)

# ---------- formatting helpers ----------
def hex1_u4(v: int) -> str:
    return f"{(v & 0xF):01x}"

def pack_u4_32(vec_u8: np.ndarray) -> str:
    return "".join(f"{int(x) & 0xF:01x}" for x in vec_u8)

def pack_i4_32(vec_i8: np.ndarray) -> str:
    return "".join(f"{int(x) & 0xF:01x}" for x in vec_i8)

def pack_i16_32(vec_i16: np.ndarray) -> str:
    u16 = vec_i16.view(np.uint16)
    return "".join(f"{int(x):04x}" for x in u16)

def ensure_len32(vec: np.ndarray, pad_value: int, dtype) -> np.ndarray:
    out = np.full((PE_NUM,), pad_value, dtype=dtype)
    n = min(PE_NUM, vec.shape[0])
    out[:n] = vec[:n].astype(dtype, copy=False)
    return out

def _write_line(out_f, s: str):
    if out_f is None: print(s)
    else: out_f.write(s + "\n")

def make_step_tag(layer_name: str, group_idx_1based: int, group_cnt: int, k_1based: int, L: int) -> str:
    fc = layer_name.replace("fc", "FC")
    pad = len(str(L))
    k_str = str(k_1based).zfill(pad)
    return f"[{fc}({group_idx_1based}/{group_cnt}), acc({k_str}/{L})]"

def maybe_rev32(vec: np.ndarray, reverse32: bool) -> np.ndarray:
    return vec[::-1] if reverse32 else vec

# ---------- core dump for one layer (4-bit) ----------
def dump_layer_cyclewise(
    layer_name: str,
    x_u4: np.ndarray,     # [in_ch], uint8 (0..15)
    layer_meta: dict,     # meta[layer_name]
    W: np.ndarray,        # [out_ch, in_ch], int8 (-8..7)
    b: np.ndarray,        # [out_ch], int16
    in_zp: int,           # Passed from previous layer
    show: dict,           # keys: x,wvec,mul,acc,plus_bias,clip,quant, reverse32
    k_range=None,
    group_bases=None,
    out_f=None,
    header=True,
):
    out_ch, in_ch = map(int, layer_meta["w_shape"])
    assert x_u4.shape[0] == in_ch
    assert W.shape == (out_ch, in_ch)
    assert b.shape == (out_ch,)

    out_zp = int(layer_meta.get("out_zp", 0))
    rq_s   = int(layer_meta["rq_scale_int12"])
    rq_sh  = int(layer_meta["rq_shift_uint4"])
    reverse32 = bool(show.get("reverse32", False))

    # Determine full calculation range
    full_k_list = range(in_ch)
    
    # Determine print range (subset of full range)
    print_k_list = k_range if k_range is not None else full_k_list

    all_group_bases = list(range(0, out_ch, PE_NUM))
    group_cnt = len(all_group_bases)
    if group_bases is None: group_bases = all_group_bases

    if header:
        _write_line(out_f, "")
        _write_line(out_f, "#"*120)
        _write_line(out_f, f"# LAYER={layer_name}  in_ch={in_ch} out_ch={out_ch} groups={all_group_bases}")
        _write_line(out_f, f"# in_zp={in_zp} out_zp={out_zp} rq_s={rq_s} rq_shift={rq_sh}")
        _write_line(out_f, "#"*120)

    y_out = np.zeros((out_ch,), dtype=np.uint8)

    for out_base in group_bases:
        group_idx_1based = (out_base // PE_NUM) + 1
        valid = min(PE_NUM, out_ch - out_base)

        b_vec = b[out_base:out_base+valid].astype(np.int16, copy=False)
        b_pad = ensure_len32(b_vec, 0, np.int16)

        acc16 = np.zeros((PE_NUM,), dtype=np.int16)
        last_quant_u4 = np.zeros((PE_NUM,), dtype=np.uint8)

        _write_line(out_f, "")
        _write_line(out_f, f"# --- {layer_name} out_base={out_base} (group {group_idx_1based}/{group_cnt}, valid lanes={valid}) ---")

        # 【重要】必ず全サイクル計算する
        for k in full_k_list:
            xk_val = int(x_u4[k])

            # --- MAC Logic ---
            x_center = np.int16(xk_val) - np.int16(in_zp)
            w_vec = W[out_base:out_base+valid, k]
            w_pad = ensure_len32(w_vec, 0, np.int8)
            mul16 = (x_center * w_pad.astype(np.int16)).astype(np.int16)
            acc16 = sat_add_i16_vec(acc16, mul16)

            # --- Output Logic ---
            sum_sat_i16 = sat_add_i16_vec(acc16, b_pad)
            rq_i32 = requant_hw_4bit(sum_sat_i16, rq_s, rq_sh)
            q_u4 = clip_to_u4_vec(rq_i32, out_zp)
            last_quant_u4 = q_u4

            # --- Conditional Print ---
            if k in print_k_list:
                k1 = int(k) + 1
                tag = make_step_tag(layer_name, group_idx_1based, group_cnt, k1, in_ch)
                
                w_disp   = maybe_rev32(w_pad, reverse32)
                mul_disp = maybe_rev32(mul16, reverse32)
                acc_disp = maybe_rev32(acc16, reverse32)
                pb_disp  = maybe_rev32(sum_sat_i16, reverse32)
                q_disp   = maybe_rev32(q_u4, reverse32)

                parts = [tag]
                if show.get("x", False): parts.append(hex1_u4(xk_val))
                if show.get("wvec", False): parts.append(pack_i4_32(w_disp))
                if show.get("mul", False): parts.append(pack_i16_32(mul_disp))
                if show.get("acc", False): parts.append(pack_i16_32(acc_disp))
                if show.get("plus_bias", False): parts.append(pack_i16_32(pb_disp))
                if show.get("quant", False): parts.append(pack_u4_32(q_disp))

                _write_line(out_f, " ".join(parts))

        y_out[out_base:out_base+valid] = last_quant_u4[:valid]

    return y_out

# ---------- MLP runner ----------
def dump_mlp_cyclewise(
    export_dir: Path,
    input_hex: Path,
    show: dict,
    out_f=None,
    k_range_override: dict | None = None,
    group_base_override: dict | None = None,
    expected_label: int | None = None,
):
    meta = load_meta(export_dir / "meta.json")
    x0_u4 = load_u4_hex_n(input_hex, 196)

    def load_layer_params(name: str):
        out_ch, in_ch = map(int, meta[name]["w_shape"])
        W = load_weight_rowmajor_int4(export_dir / f"{name}_w_int4.hex", out_ch, in_ch)
        b = load_bias_int16(export_dir / f"{name}_b_int16.hex", out_ch)
        return W, b

    W1, b1 = load_layer_params("fc1")
    W2, b2 = load_layer_params("fc2")
    W3, b3 = load_layer_params("fc3")

    k_over = k_range_override or {}
    g_over = group_base_override or {}

    y1_u4 = dump_layer_cyclewise(
        "fc1", x0_u4, meta["fc1"], W1, b1, 0,
        show=show,
        k_range=k_over.get("fc1", None),
        group_bases=g_over.get("fc1", None),
        out_f=out_f, header=True
    )

    y2_u4 = dump_layer_cyclewise(
        "fc2", y1_u4, meta["fc2"], W2, b2, int(meta["fc1"].get("out_zp", 0)),
        show=show,
        k_range=k_over.get("fc2", None),
        group_bases=g_over.get("fc2", None),
        out_f=out_f, header=True
    )

    y3_u4 = dump_layer_cyclewise(
        "fc3", y2_u4, meta["fc3"], W3, b3, int(meta["fc2"].get("out_zp", 0)),
        show=show,
        k_range=k_over.get("fc3", None),
        group_bases=g_over.get("fc3", None),
        out_f=out_f, header=True
    )

    pred = int(np.argmax(y3_u4[:10]))
    _write_line(out_f, "")
    _write_line(out_f, "="*120)
    _write_line(out_f, f"# FINAL y3_u4 (0..9) = {[int(v) for v in y3_u4[:10]]}")
    _write_line(out_f, f"# PRED = {pred}")
    if expected_label is not None:
        ok = (pred == int(expected_label))
        _write_line(out_f, f"# EXPECTED_LABEL = {int(expected_label)}  -> {'OK' if ok else 'NG'}")
    _write_line(out_f, "="*120)

    return dict(y1=y1_u4, y2=y2_u4, y3=y3_u4, pred=pred)

In [6]:
# ============================================================
# [Jupyter Cell] サイクル単位デバッグ実行スクリプト
# ※ 前のセルで dump_mlp_cyclewise 等が定義されている前提です
# ============================================================
from pathlib import Path
import sys

# 1. パス設定 (エクスポートフォルダ)
EXPORT_DIR = Path(r"../export_4bit_asic")
INPUT_DIR  = EXPORT_DIR / "inputs" / "correct"

# 2. テスト対象のHEXファイルを自動取得 (最初に見つかったもの)
if not INPUT_DIR.exists():
    print(f"Error: '{INPUT_DIR}' が見つかりません。エクスポートを実行してください。")
else:
    hex_files = list(INPUT_DIR.glob("*.hex"))
    if not hex_files:
        print("Error: テスト用HEXファイルが見つかりません。")
        INPUT_HEX = None
    else:
        INPUT_HEX = hex_files[0]
        print(f"Target Input: {INPUT_HEX.name}")

# 3. 正解ラベルの推定 (ファイル名から)
EXPECTED_LABEL = None
if INPUT_HEX:
    try:
        # 例: mnist_0000_label7_pred7.hex -> 7
        parts = INPUT_HEX.stem.split("_")
        label_part = [p for p in parts if p.startswith("label")][0]
        EXPECTED_LABEL = int(label_part.replace("label", ""))
    except:
        pass

# ============================================================
# 表示オプション設定
# ============================================================
# 表示したい項目を True にしてください
SHOW_OPTS = dict(
    reverse32   = True,   # True: Lane 31..0 (波形比較用), False: Lane 0..31
    x           = False,   # Input value (uint4)
    wvec        = False,   # Weight vector (int4)
    mul         = False,  # Multiply result (int16) -> 冗長なので非表示
    acc         = True,   # Accumulator (int16) -> 重要
    plus_bias   = False,   # Acc + Bias (Saturating int16) -> 重要 (Requant入力)
    quant       = False,   # Final Output (uint4) -> 重要
)

# ============================================================
# 出力範囲の制限 (ログが長くなりすぎないように)
# ============================================================
# None を指定すると全サイクル出力します
K_RANGE = {
    "fc1": None, # range(0, 5),    # FC1は長い(196サイクル)ので、最初の5サイクルだけ確認
    "fc2": None,           # FC2(32サイクル)は全部見る
    "fc3": None,           # FC3(16サイクル)は全部見る
}

# 表示する並列グループ (FC1はOut32なのでGroup0のみ)
GROUP_BASE = {
    "fc1": [0], 
    "fc2": [0],
    "fc3": [0],
}

# ============================================================
# 実行 & 表示
# ============================================================
if INPUT_HEX:
    print("-" * 60)
    print(f"Running Cycle-Accurate Dump for {INPUT_HEX.name}...")
    print("-" * 60)
    
    # 出力先: Noneならこの画面(標準出力)に出す
    # ファイルに残したい場合は: out_f = open("debug_log.txt", "w", encoding="utf-8")
    out_f = None 

    try:
        # 関数呼び出し
        result = dump_mlp_cyclewise(
            export_dir=EXPORT_DIR,
            input_hex=INPUT_HEX,
            show=SHOW_OPTS,
            out_f=out_f,
            k_range_override=K_RANGE,
            group_base_override=GROUP_BASE,
            expected_label=EXPECTED_LABEL
        )
        
        # 最終結果表示
        print("\n" + "="*60)
        print(f" Prediction: {result['pred']}")
        if EXPECTED_LABEL is not None:
            is_ok = (result['pred'] == EXPECTED_LABEL)
            print(f" Correct:    {'YES' if is_ok else 'NO'} (Label: {EXPECTED_LABEL})")
        print(f" Final Logits (uint4): {list(result['y3'][:10])}")
        print("="*60)
        
    except Exception as e:
        print(f"\nError occurred: {e}")
        import traceback
        traceback.print_exc()

Target Input: mnist_0000_label7_pred7.hex
------------------------------------------------------------
Running Cycle-Accurate Dump for mnist_0000_label7_pred7.hex...
------------------------------------------------------------

########################################################################################################################
# LAYER=fc1  in_ch=196 out_ch=32 groups=[0]
# in_zp=0 out_zp=0 rq_s=396 rq_shift=15
########################################################################################################################

# --- fc1 out_base=0 (group 1/1, valid lanes=32) ---
[FC1(1/1), acc(001/196)] 00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
[FC1(1/1), acc(002/196)] 00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
[FC1(1/1), acc(003/196)] 00000000000000000000000000000000000000000000000000000000000